# Validation Notebook — FSDD Spoken Digit Classifiers

This notebook is the **final acceptance test** for all four trained models.
Every cell either prints `PASS` or raises `AssertionError` with a descriptive message.

| Check | What it verifies |
|-------|------------------|
| 0 | Imports and checkpoint existence |
| 1 | Checkpoint integrity (non-empty, no NaN/Inf, key shapes) |
| 2 | Per-layer ≤ 36 kB memory constraint |
| 3 | INT8 weight snap verification (B2) |
| 4 | Power-of-Two weight verification + distribution chart (C) |
| 5 | Forward pass smoke test |
| 6 | Full test-set inference + confusion matrix |
| 7 | Inference speed benchmark |
| 8 | Consolidated PASS/FAIL summary |

**Before running:** ensure all four training scripts have completed and their checkpoints exist in the project root.

In [ ]:
# ── Cell 0: Imports and Setup ────────────────────────────────────────────────
import os, sys, time, copy
import warnings; warnings.filterwarnings('ignore')
from collections import Counter

import numpy as np
import torch
import matplotlib.pyplot as plt

# Notebook lives in notebooks/; add project root to path
sys.path.insert(0, os.path.abspath('..'))
os.chdir(os.path.abspath('..'))

# Always run validation on CPU for reproducibility
DEVICE = torch.device('cpu')

# Actual checkpoint filenames from config.py
CHECKPOINTS = {
    'A':  'best_model.pt',
    'B1': 'best_model_b1_constrained.pt',
    'B2': 'best_model_b2_int8.pt',
    'C':  'best_model_c_pow2.pt',
}

os.makedirs('figures', exist_ok=True)

print(f"PyTorch version : {torch.__version__}")
print(f"Working directory: {os.getcwd()}")
print()
print("Checkpoint status:")
for name, path in CHECKPOINTS.items():
    exists = os.path.exists(path)
    size   = f"{os.path.getsize(path)/1024:.1f} kB" if exists else "—"
    status = "FOUND" if exists else "MISSING"
    print(f"  Task {name}: {status:7s}  {size:>10}  ({path})")

print("\nImports OK")

## Check 1 — Checkpoint Integrity

For each checkpoint: verify it exists, is non-empty, loads as a valid `state_dict`, and contains no `NaN` or `Inf` values in any tensor.

In [ ]:
# ── Cell 1: Checkpoint Integrity ────────────────────────────────────────────
_integrity_results = {}

def check_checkpoint_integrity(path, task_name):
    print(f"\n{'='*60}")
    print(f"  Checkpoint integrity: Task {task_name} — {path}")
    print(f"{'='*60}")

    assert os.path.exists(path), f"FAIL: {path} not found"
    size_kb = os.path.getsize(path) / 1024
    print(f"  File size : {size_kb:.1f} kB")
    assert size_kb > 1, f"FAIL: file suspiciously small ({size_kb:.1f} kB)"

    state = torch.load(path, map_location='cpu')
    assert isinstance(state, dict), "FAIL: not a state_dict (expected dict)"

    total_params = 0
    has_bad = False
    print(f"  {'Key':<45} {'Shape':<25} dtype")
    print(f"  {'-'*45} {'-'*25} -----")
    for k, v in sorted(state.items()):
        if not torch.is_tensor(v):
            continue
        n = v.numel()
        total_params += n
        bad = torch.isnan(v).any().item() or torch.isinf(v).any().item()
        if bad:
            has_bad = True
            print(f"  {k:<45} {str(tuple(v.shape)):<25} {v.dtype}  *** NaN/Inf ***")
        else:
            print(f"  {k:<45} {str(tuple(v.shape)):<25} {v.dtype}")

    assert not has_bad, "FAIL: NaN or Inf values detected in checkpoint"
    print(f"\n  Total parameters : {total_params:,}")
    print(f"  PASS  Checkpoint integrity OK")
    return state, total_params


_states = {}
for task, path in CHECKPOINTS.items():
    if os.path.exists(path):
        try:
            state, n = check_checkpoint_integrity(path, task)
            _states[task] = state
            _integrity_results[task] = 'PASS'
        except AssertionError as e:
            print(f"  FAIL: {e}")
            _integrity_results[task] = 'FAIL'
    else:
        print(f"\n  Skipping Task {task} — checkpoint not found")
        _integrity_results[task] = 'SKIP'

## Check 2 — Per-Layer Memory Constraint

Verify that every layer of Tasks B1, B2, and C fits within the 36 kB per-layer SRAM constraint.
- **B1**: float32, bytes_per_param = 4, limit = 9,000 params
- **B2/C**: INT8, bytes_per_param = 1, limit = 36,000 params

In [ ]:
# ── Cell 2: Per-Layer Memory Constraint ────────────────────────────────────
from main import DigitGRU
from task_b1_constrained import ConstrainedMGU

_memory_results = {}
LIMIT_BYTES = 36_000


def check_layer_memory(model, state_dict, task_name, bytes_per_param):
    model.load_state_dict(state_dict, strict=False)
    model.eval()
    print(f"\n{'='*70}")
    print(f"  Layer memory: Task {task_name}  |  {bytes_per_param} byte(s)/param  |  limit={LIMIT_BYTES} B")
    print(f"{'='*70}")
    print(f"  {'Layer':<35} {'Params':>8}  {'Bytes':>9}  {'Limit':>8}  Status")
    print(f"  {'-'*35} {'-'*8}  {'-'*9}  {'-'*8}  ------")

    violations = []
    for name, module in model.named_modules():
        params = list(module.parameters(recurse=False))
        if not params:
            continue
        n      = sum(p.numel() for p in params)
        n_bytes = n * bytes_per_param
        ok     = n_bytes <= LIMIT_BYTES
        status = "OK" if ok else "EXCEEDS LIMIT"
        if not ok:
            violations.append((name, n_bytes))
        print(f"  {name:<35} {n:>8,}  {n_bytes:>8,} B  {LIMIT_BYTES:>8,}  {status}")

    assert not violations, f"FAIL: layers exceed {LIMIT_BYTES} B: {violations}"
    print(f"\n  PASS  All layers within {LIMIT_BYTES} B constraint")


# Task A — no constraint, just show total size
if 'A' in _states:
    model_a = DigitGRU(input_size=120, hidden_size=128)
    model_a.load_state_dict(_states['A'])
    total_a = sum(p.numel() for p in model_a.parameters())
    print(f"Task A — total params: {total_a:,} (no per-layer constraint)")
    _memory_results['A'] = 'N/A'

# Task B1 — float32 constraint
if 'B1' in _states:
    try:
        model_b1 = ConstrainedMGU(n_mfcc=13)
        check_layer_memory(model_b1, _states['B1'], 'B1', bytes_per_param=4)
        _memory_results['B1'] = 'PASS'
    except AssertionError as e:
        print(f"  FAIL: {e}"); _memory_results['B1'] = 'FAIL'
else:
    _memory_results['B1'] = 'SKIP'

# Task B2 — INT8 constraint
if 'B2' in _states:
    try:
        from task_b2_int8 import QuantizedConstrainedMGU
        model_b2 = QuantizedConstrainedMGU(n_mfcc=13)
        check_layer_memory(model_b2, _states['B2'], 'B2', bytes_per_param=1)
        _memory_results['B2'] = 'PASS'
    except AssertionError as e:
        print(f"  FAIL: {e}"); _memory_results['B2'] = 'FAIL'
    except FileNotFoundError:
        print("  Skipping Task B2 — checkpoint not found"); _memory_results['B2'] = 'SKIP'
else:
    _memory_results['B2'] = 'SKIP'

# Task C — INT8 constraint
if 'C' in _states:
    try:
        from task_c_pow2 import PoTConstrainedMGU
        model_c = PoTConstrainedMGU(n_mfcc=13)
        check_layer_memory(model_c, _states['C'], 'C', bytes_per_param=1)
        _memory_results['C'] = 'PASS'
    except AssertionError as e:
        print(f"  FAIL: {e}"); _memory_results['C'] = 'FAIL'
    except FileNotFoundError:
        print("  Skipping Task C — checkpoint not found"); _memory_results['C'] = 'SKIP'
else:
    _memory_results['C'] = 'SKIP'

## Check 3 — INT8 Weight Snap Verification (Task B2)

After INT8 snapping, all weight values should lie on the INT8 grid:
`round(w / scale) == w / scale` within floating-point tolerance,  
where `scale = max(|w|) / 127` per tensor.

In [ ]:
# ── Cell 3: INT8 Weight Snap Verification ──────────────────────────────────
_int8_results = {}

def check_int8_weights(state_dict, task_name):
    """After INT8 snap, w/scale must be integer-valued (tolerance 0.01)."""
    print(f"\n{'='*60}")
    print(f"  INT8 weight verification: Task {task_name}")
    print(f"{'='*60}")
    print(f"  {'Key':<45} {'max_frac':>10}  Status")
    print(f"  {'-'*45} {'-'*10}  ------")

    all_ok = True
    for k, v in sorted(state_dict.items()):
        if 'weight' not in k or not torch.is_tensor(v):
            continue
        scale = v.detach().abs().max() / 127.0
        if scale < 1e-12:
            print(f"  {k:<45} {'(zero)':>10}  OK")
            continue
        v_scaled     = v / scale
        max_frac     = (v_scaled - v_scaled.round()).abs().max().item()
        ok           = max_frac < 0.01
        if not ok:
            all_ok = False
        print(f"  {k:<45} {max_frac:>10.6f}  {'OK' if ok else 'FAIL'}")

    assert all_ok, "FAIL: B2 weights contain non-INT8-quantized values"
    print(f"\n  PASS  All B2 weight tensors are on the INT8 grid")


if 'B2' in _states:
    try:
        from utils.quant_utils import FakeQuantizeLinear, snap_weights_to_int8
        # Load the saved B2 weights (these are the fake-quant training weights;
        # we snap a copy to verify the snapped values are truly INT8-aligned)
        model_b2 = QuantizedConstrainedMGU(n_mfcc=13)
        model_b2.load_state_dict(_states['B2'], strict=False)
        snapped_b2 = copy.deepcopy(model_b2)
        snap_weights_to_int8(snapped_b2)
        check_int8_weights(snapped_b2.state_dict(), 'B2')
        _int8_results['B2'] = 'PASS'
    except AssertionError as e:
        print(f"  FAIL: {e}"); _int8_results['B2'] = 'FAIL'
    except Exception as e:
        print(f"  ERROR: {e}"); _int8_results['B2'] = 'ERROR'
else:
    print("Skipping Check 3 — Task B2 checkpoint not available")
    _int8_results['B2'] = 'SKIP'

## Check 4 — Power-of-Two Weight Verification (Task C)

After PoT snap, every non-zero weight must satisfy:
- `log₂|w|` is an integer (within tolerance)
- The exponent is in `[POT_MIN_EXP, POT_MAX_EXP] = [-8, 3]`

Also plots the weight exponent distribution.

In [ ]:
# ── Cell 4: PoT Weight Verification ────────────────────────────────────────
_pot_results = {}
POT_MIN_EXP, POT_MAX_EXP = -8, 3

def check_pot_weights(state_dict, task_name):
    """All non-zero weights must be ±2^k with k in [POT_MIN_EXP, POT_MAX_EXP]."""
    print(f"\n{'='*65}")
    print(f"  Power-of-Two weight verification: Task {task_name}")
    print(f"{'='*65}")
    print(f"  {'Key':<45} {'log2_frac':>10}  {'zeros%':>7}  Status")
    print(f"  {'-'*45} {'-'*10}  {'-'*7}  ------")

    violations = []
    all_weights_flat = []

    for k, v in sorted(state_dict.items()):
        if 'weight' not in k or not torch.is_tensor(v):
            continue

        flat    = v.flatten()
        nonzero = flat[flat.abs() > 1e-9]
        zero_pct = (flat.abs() <= 1e-9).float().mean().item() * 100
        all_weights_flat.extend(flat.tolist())

        if len(nonzero) == 0:
            print(f"  {k:<45} {'(all zero)':>10}  {zero_pct:>6.1f}%  OK")
            continue

        log2v    = torch.log2(nonzero.abs())
        max_frac = (log2v - log2v.round()).abs().max().item()
        exps     = log2v.round()
        in_range = bool((exps >= POT_MIN_EXP).all() and (exps <= POT_MAX_EXP).all())
        ok       = (max_frac < 0.01) and in_range
        if not ok:
            violations.append(k)

        print(f"  {k:<45} {max_frac:>10.4f}  {zero_pct:>6.1f}%  {'OK' if ok else 'FAIL'}")

    assert not violations, f"FAIL: non-PoT weights in: {violations}"
    print(f"\n  PASS  All Task C weights are valid powers of two")

    # ── Weight exponent distribution chart ──
    arr     = np.array(all_weights_flat)
    nonzero = arr[np.abs(arr) > 1e-9]
    zero_n  = int((np.abs(arr) <= 1e-9).sum())

    if len(nonzero) > 0:
        exps        = np.round(np.log2(np.abs(nonzero))).astype(int)
        exp_counts  = Counter(exps.tolist())
        exp_range   = list(range(POT_MIN_EXP, POT_MAX_EXP + 1))
        counts      = [exp_counts.get(e, 0) for e in exp_range]
        total       = len(arr)

        fig, ax = plt.subplots(figsize=(11, 4))
        bars = ax.bar(
            [str(e) for e in exp_range], counts,
            color='steelblue', edgecolor='white', linewidth=0.5,
        )
        ax.set_xlabel('Exponent  k  (weight magnitude = 2^k)', fontsize=11)
        ax.set_ylabel('Count (positive + negative combined)', fontsize=11)
        ax.set_title(
            f'Task C — PoT Weight Exponent Distribution\n'
            f'({total:,} total weights   |   {zero_n:,} zeros = {zero_n/total*100:.1f}%)',
            fontsize=12,
        )
        for bar, cnt in zip(bars, counts):
            if cnt > 0:
                ax.text(
                    bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                    str(cnt), ha='center', va='bottom', fontsize=8,
                )
        plt.tight_layout()
        out_path = 'figures/task_c_weight_dist_validation.png'
        plt.savefig(out_path, dpi=150)
        plt.show()
        print(f"  Chart saved to {out_path}")


if 'C' in _states:
    try:
        from task_c_pow2 import PoTConstrainedMGU
        from utils.quant_utils import snap_weights_to_pot
        model_c = PoTConstrainedMGU(n_mfcc=13)
        model_c.load_state_dict(_states['C'], strict=False)
        snapped_c = copy.deepcopy(model_c)
        snap_weights_to_pot(snapped_c)
        check_pot_weights(snapped_c.state_dict(), 'C')
        _pot_results['C'] = 'PASS'
    except AssertionError as e:
        print(f"  FAIL: {e}"); _pot_results['C'] = 'FAIL'
    except Exception as e:
        print(f"  ERROR: {e}"); _pot_results['C'] = 'ERROR'
else:
    print("Skipping Check 4 — Task C checkpoint not available")
    _pot_results['C'] = 'SKIP'

## Check 5 — Forward Pass Smoke Test

For each model: run a single forward pass with a synthetic random batch of shape `(4, 100, input_size)` and verify:
- Output shape is `(4, 10)` ✓
- No NaN or Inf in the output ✓

In [ ]:
# ── Cell 5: Forward Pass Smoke Test ────────────────────────────────────────
_smoke_results = {}

def smoke_test(model, input_size, task_name):
    model.eval()
    print(f"  Smoke test Task {task_name} (input_size={input_size}) ... ", end="")
    with torch.no_grad():
        x      = torch.randn(4, 100, input_size)
        logits = model(x)
    assert logits.shape == (4, 10), \
        f"FAIL: expected shape (4, 10), got {logits.shape}"
    assert not torch.isnan(logits).any(), "FAIL: NaN in output logits"
    assert not torch.isinf(logits).any(), "FAIL: Inf in output logits"
    rng = f"[{logits.min():.3f}, {logits.max():.3f}]"
    print(f"shape={logits.shape}  logit_range={rng}  PASS")


print("Forward pass smoke tests:")
print()

if 'A' in _states:
    try:
        smoke_test(model_a, input_size=120, task_name='A')
        _smoke_results['A'] = 'PASS'
    except AssertionError as e:
        print(f"FAIL: {e}"); _smoke_results['A'] = 'FAIL'
else:
    _smoke_results['A'] = 'SKIP'

if 'B1' in _states:
    try:
        smoke_test(model_b1, input_size=39, task_name='B1')
        _smoke_results['B1'] = 'PASS'
    except AssertionError as e:
        print(f"FAIL: {e}"); _smoke_results['B1'] = 'FAIL'
else:
    _smoke_results['B1'] = 'SKIP'

if 'B2' in _states:
    try:
        smoke_test(model_b2, input_size=39, task_name='B2')
        _smoke_results['B2'] = 'PASS'
    except (AssertionError, Exception) as e:
        print(f"FAIL: {e}"); _smoke_results['B2'] = 'FAIL'
else:
    print("  Skipping Task B2 — not available")
    _smoke_results['B2'] = 'SKIP'

if 'C' in _states:
    try:
        smoke_test(model_c, input_size=39, task_name='C')
        _smoke_results['C'] = 'PASS'
    except (AssertionError, Exception) as e:
        print(f"FAIL: {e}"); _smoke_results['C'] = 'FAIL'
else:
    print("  Skipping Task C — not available")
    _smoke_results['C'] = 'SKIP'

## Check 6 — Full Test-Set Inference

Build the test DataLoader using `random_state=42` (same split as training) and run full inference for each available model. Reports test accuracy, per-class accuracy bar chart, and confusion matrix heatmap.

Accuracy thresholds:
```
A  ≥ 0.95   B1 ≥ 0.88   B2 ≥ 0.85   C ≥ 0.70
```

In [ ]:
# ── Cell 6: Full Test-Set Inference ────────────────────────────────────────
from utils.data_loader import build_loaders
import sklearn.metrics as skm

_inference_results = {}

MIN_ACC = {'A': 0.95, 'B1': 0.88, 'B2': 0.85, 'C': 0.70}

print("Building test loaders (this may take ~30 s on first run) ...")
_, _, test_loader_a,  feat_a  = build_loaders(n_mfcc=40, random_state=42)
_, _, test_loader_b1, feat_b1 = build_loaders(n_mfcc=13, random_state=42)
print(f"  Task A  feature_dim={feat_a}")
print(f"  B1/B2/C feature_dim={feat_b1}")
print()


def run_inference(model, loader, task_name, min_acc):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, y in loader:
            preds = model(x.to(DEVICE)).argmax(1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(y.tolist())

    acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
    print(f"Task {task_name} — test accuracy: {acc:.4f} ({acc*100:.1f}%)")

    # Per-class accuracy
    print("  Per-class accuracy:")
    cc, ct = Counter(), Counter()
    for p, l in zip(all_preds, all_labels):
        ct[l] += 1
        if p == l: cc[l] += 1
    for c in range(10):
        ca  = cc[c] / ct[c] if ct[c] > 0 else 0
        bar = '█' * int(ca * 20)
        print(f"    Digit {c}: {ca:.2f}  {bar}")

    # Confusion matrix
    cm = skm.confusion_matrix(all_labels, all_preds)
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(10)); ax.set_yticks(range(10))
    ax.set_xticklabels(range(10)); ax.set_yticklabels(range(10))
    ax.set_xlabel('Predicted digit'); ax.set_ylabel('True digit')
    ax.set_title(f'Task {task_name} — Confusion Matrix (acc={acc:.3f})')
    threshold = cm.max() / 2
    for i in range(10):
        for j in range(10):
            ax.text(j, i, cm[i, j], ha='center', va='center',
                    color='white' if cm[i, j] > threshold else 'black', fontsize=8)
    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    out = f'figures/task_{task_name.lower()}_confusion.png'
    plt.savefig(out, dpi=150)
    plt.show()
    print(f"  Confusion matrix saved to {out}")

    assert acc >= min_acc, \
        f"FAIL: Task {task_name} accuracy {acc:.4f} < minimum {min_acc}"
    print(f"  PASS  accuracy {acc:.4f} >= {min_acc}")
    return acc


if 'A' in _states:
    try:
        acc = run_inference(model_a, test_loader_a, 'A', MIN_ACC['A'])
        _inference_results['A'] = f'PASS ({acc:.3f})'
    except AssertionError as e:
        print(f"  FAIL: {e}"); _inference_results['A'] = 'FAIL'
else:
    _inference_results['A'] = 'SKIP'

if 'B1' in _states:
    try:
        acc = run_inference(model_b1, test_loader_b1, 'B1', MIN_ACC['B1'])
        _inference_results['B1'] = f'PASS ({acc:.3f})'
    except AssertionError as e:
        print(f"  FAIL: {e}"); _inference_results['B1'] = 'FAIL'
else:
    _inference_results['B1'] = 'SKIP'

if 'B2' in _states:
    try:
        acc = run_inference(model_b2, test_loader_b1, 'B2', MIN_ACC['B2'])
        _inference_results['B2'] = f'PASS ({acc:.3f})'
    except AssertionError as e:
        print(f"  FAIL: {e}"); _inference_results['B2'] = 'FAIL'
    except Exception as e:
        print(f"  ERROR: {e}"); _inference_results['B2'] = 'ERROR'
else:
    print("Skipping Task B2 inference — checkpoint not available")
    _inference_results['B2'] = 'SKIP'

if 'C' in _states:
    try:
        acc = run_inference(model_c, test_loader_b1, 'C', MIN_ACC['C'])
        _inference_results['C'] = f'PASS ({acc:.3f})'
    except AssertionError as e:
        print(f"  FAIL: {e}"); _inference_results['C'] = 'FAIL'
    except Exception as e:
        print(f"  ERROR: {e}"); _inference_results['C'] = 'ERROR'
else:
    print("Skipping Task C inference — checkpoint not available")
    _inference_results['C'] = 'SKIP'

## Check 7 — Inference Speed Benchmark (CPU)

Measure average single-sample inference latency on CPU. This is the most relevant metric for edge deployment scenarios.

In [ ]:
# ── Cell 7: Inference Speed Benchmark ──────────────────────────────────────
_speed_results = {}
N_WARMUP = 10
N_RUNS   = 200

def benchmark(model, input_size, task_name):
    model.eval()
    x = torch.randn(1, 100, input_size)
    # Warmup
    with torch.no_grad():
        for _ in range(N_WARMUP):
            _ = model(x)
    # Timed runs
    t0 = time.perf_counter()
    with torch.no_grad():
        for _ in range(N_RUNS):
            _ = model(x)
    elapsed   = time.perf_counter() - t0
    ms_sample = elapsed / N_RUNS * 1000
    print(f"  Task {task_name}: {ms_sample:.3f} ms/sample  ({N_RUNS} runs)")
    return ms_sample


print(f"Inference speed benchmark — CPU, single sample, {N_RUNS} runs:")
print()

times = {}
if 'A' in _states:
    try:
        times['A'] = benchmark(model_a, 120, 'A')
        _speed_results['A'] = 'PASS'
    except Exception as e:
        print(f"  ERROR Task A: {e}"); _speed_results['A'] = 'ERROR'
else:
    _speed_results['A'] = 'SKIP'

if 'B1' in _states:
    try:
        times['B1'] = benchmark(model_b1, 39, 'B1')
        _speed_results['B1'] = 'PASS'
    except Exception as e:
        print(f"  ERROR Task B1: {e}"); _speed_results['B1'] = 'ERROR'
else:
    _speed_results['B1'] = 'SKIP'

if 'B2' in _states:
    try:
        times['B2'] = benchmark(model_b2, 39, 'B2')
        _speed_results['B2'] = 'PASS'
    except Exception as e:
        print(f"  Skipping B2: {e}"); _speed_results['B2'] = 'SKIP'
else:
    print("  Skipping Task B2 — not available")
    _speed_results['B2'] = 'SKIP'

if 'C' in _states:
    try:
        times['C'] = benchmark(model_c, 39, 'C')
        _speed_results['C'] = 'PASS'
    except Exception as e:
        print(f"  Skipping C: {e}"); _speed_results['C'] = 'SKIP'
else:
    print("  Skipping Task C — not available")
    _speed_results['C'] = 'SKIP'

print()
if 'A' in times and 'B1' in times:
    print(f"Speedup B1 vs A (single-sample, CPU): {times['A']/times['B1']:.1f}x")
if 'B1' in times and 'B2' in times:
    print(f"Speedup B2 vs B1:                     {times['B1']/times['B2']:.1f}x")

## Check 8 — Consolidated PASS/FAIL Summary

Aggregates all check results from the cells above.

In [ ]:
# ── Cell 8: Consolidated Summary ────────────────────────────────────────────
TASKS = ['A', 'B1', 'B2', 'C']

checks = [
    ("Checkpoint exists & non-empty",  _integrity_results),
    ("No NaN/Inf in weights",          _integrity_results),
    ("Per-layer ≤ 36 kB constraint",   _memory_results),
    ("INT8 weight snap verified",       {k: _int8_results.get(k, 'N/A') for k in TASKS}),
    ("PoT weight snap verified",        {k: _pot_results.get(k, 'N/A') for k in TASKS}),
    ("Forward pass (correct shape)",   _smoke_results),
    ("Test accuracy ≥ threshold",       _inference_results),
    ("Inference speed benchmarked",     _speed_results),
]

# N/A overrides for tasks where a check is inapplicable by design
na_map = {
    "Per-layer ≤ 36 kB constraint":  {'A': 'N/A'},
    "INT8 weight snap verified":      {'A': 'N/A', 'B1': 'N/A', 'C': 'N/A'},
    "PoT weight snap verified":       {'A': 'N/A', 'B1': 'N/A', 'B2': 'N/A'},
}

col_w   = [42, 8, 8, 8, 8]
sep     = '+' + '+'.join('-' * w for w in col_w) + '+'
hdr_fmt = '| {:<40} | {:^6} | {:^6} | {:^6} | {:^6} |'
row_fmt = '| {:<40} | {:^6} | {:^6} | {:^6} | {:^6} |'

print("\n" + "=" * 70)
print("  VALIDATION SUMMARY")
print("=" * 70)
print(hdr_fmt.format("Check", "A", "B1", "B2", "C"))
print("-" * 70)

any_fail = False
for check_name, result_dict in checks:
    row = []
    na_overrides = na_map.get(check_name, {})
    for t in TASKS:
        val = na_overrides.get(t) or result_dict.get(t, 'SKIP')
        # Shorten 'PASS (0.xxx)' to 'PASS' for table
        short = val.split()[0] if val else 'SKIP'
        row.append(short)
        if short == 'FAIL':
            any_fail = True
    print(row_fmt.format(check_name, *row))

print("=" * 70)

if any_fail:
    print("\n  *** OVERALL: FAIL — see FAIL rows above ***")
else:
    skipped = any(
        result_dict.get(t, 'SKIP') == 'SKIP'
        for _, result_dict in checks
        for t in TASKS
    )
    if skipped:
        print("\n  OVERALL: PASS (with SKIP for untrained models)")
        print("  Note: rows marked SKIP indicate checkpoints not yet available.")
        print("        Train B2/C and re-run this notebook to complete all checks.")
    else:
        print("\n  *** OVERALL: PASS — all checks passed for all models ***")